<a href="https://colab.research.google.com/github/Hussainasif11/flyrankinternship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hussainasif11/flyrankinternship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Warehouse connection ready")

Warehouse connection ready


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Signal 1: GSC clicks**

I checked the distribution of GSC clicks in March 2026. Most available rows have very low click counts, with 3,193,080 rows having zero clicks.

Verdict: **CONFIRMED**

I will use low clicks as one signal for the baseline rule because it gives a clear volume-based way to prioritize pages for review.


**Signal 2: GSC average position**

I checked the distribution of average search position. There are 908,354 rows with an average position of 21 or worse.

Verdict: **CONFIRMED**

I will use a position above 20 as a weak-visibility signal because it indicates that the content is not ranking strongly in search.


### Baseline rule

I will score each available GSC row using two signals.

- 0 clicks = 2 points
- 1–5 clicks = 1 point
- average position above 20 = 2 points
- average position from 11–20 = 1 point

The total score is from 0 to 4.

- 4 points → REVIEW
- 2–3 points → MONITOR
- 0–1 points → NO_ACTION

The reason code will show the main signal behind the score.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

signal1 = con.sql("""
    SELECT
        CASE
            WHEN gsc_clicks = 0 THEN '0 clicks'
            WHEN gsc_clicks <= 5 THEN '1-5 clicks'
            WHEN gsc_clicks <= 20 THEN '6-20 clicks'
            ELSE '21+ clicks'
        END AS click_bucket,
        COUNT(*) AS n
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
    GROUP BY 1
    ORDER BY
        CASE click_bucket
            WHEN '0 clicks' THEN 1
            WHEN '1-5 clicks' THEN 2
            WHEN '6-20 clicks' THEN 3
            ELSE 4
        END
""").df()

signal1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,click_bucket,n
0,0 clicks,3193080
1,1-5 clicks,399035
2,6-20 clicks,17667
3,21+ clicks,1279


In [13]:
signal2 = con.sql("""
    SELECT
        CASE
            WHEN gsc_avg_position <= 3 THEN '1-3'
            WHEN gsc_avg_position <= 10 THEN '4-10'
            WHEN gsc_avg_position <= 20 THEN '11-20'
            WHEN gsc_avg_position > 20 THEN '21+'
            ELSE 'missing'
        END AS position_bucket,
        COUNT(*) AS n
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
    GROUP BY 1
    ORDER BY
        CASE position_bucket
            WHEN '1-3' THEN 1
            WHEN '4-10' THEN 2
            WHEN '11-20' THEN 3
            WHEN '21+' THEN 4
            ELSE 5
        END
""").df()

signal2

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n
0,1-3,727362
1,4-10,1456122
2,11-20,519223
3,21+,908354


## 2. Build the ranked queue (writes the CSV)

I will rank the March 2026 rows using the baseline score from the two checked signals: GSC clicks and GSC average position.

Rows with lower clicks and weaker search position receive higher scores. The score is used to prioritize pages for review.

The queue will contain one reason code and one action label for each row. The final ranked queue will be written to `work/outputs/baseline_action_score.csv`.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

queue = con.sql("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_clicks,
        gsc_avg_position,

        (
            CASE
                WHEN gsc_clicks = 0 THEN 2
                WHEN gsc_clicks <= 5 THEN 1
                ELSE 0
            END
            +
            CASE
                WHEN gsc_avg_position > 20 THEN 2
                WHEN gsc_avg_position > 10 THEN 1
                ELSE 0
            END
        ) AS score

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )

    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
""").df()

queue["action"] = queue["score"].apply(
    lambda x: "REVIEW" if x == 4
    else "MONITOR" if x >= 2
    else "NO_ACTION"
)

queue["reason_code"] = queue.apply(
    lambda row:
        "LOW_CLICKS_AND_WEAK_POSITION"
        if row["gsc_clicks"] == 0 and row["gsc_avg_position"] > 20
        else "LOW_CLICKS"
        if row["gsc_clicks"] <= 5
        else "WEAK_POSITION"
        if row["gsc_avg_position"] > 20
        else "NO_STRONG_SIGNAL",
    axis=1
)

queue = queue.sort_values(
    ["score", "gsc_clicks"],
    ascending=[False, True]
).reset_index(drop=True)

queue["rank"] = range(1, len(queue) + 1)

queue.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_clicks,gsc_avg_position,score,action,reason_code,rank
0,2026-03-01,client_73cda7b4e4f265ea,content_476c37c366920c1b,0,41.000000,4,REVIEW,LOW_CLICKS_AND_WEAK_POSITION,1
1,2026-03-01,client_73cda7b4e4f265ea,content_f71459b346aba398,0,38.500000,4,REVIEW,LOW_CLICKS_AND_WEAK_POSITION,2
2,2026-03-01,client_73cda7b4e4f265ea,content_098eedbd77ec1de1,0,41.111111,4,REVIEW,LOW_CLICKS_AND_WEAK_POSITION,3
3,2026-03-01,client_73cda7b4e4f265ea,content_fd377b1af29dcbb3,0,20.384615,4,REVIEW,LOW_CLICKS_AND_WEAK_POSITION,4
4,2026-03-01,client_73cda7b4e4f265ea,content_61c495d39498082e,0,28.894737,4,REVIEW,LOW_CLICKS_AND_WEAK_POSITION,5
5,2026-03-01,client_73cda7b4e4f265ea,content_097e9ae329f3f6f3,0,35.400000,4,REVIEW,LOW_CLICKS_AND_WEAK_POSITION,6
6,2026-03-01,client_73cda7b4e4f265ea,content_76871bb7b214573c,0,89.000000,4,REVIEW,LOW_CLICKS_AND_WEAK_POSITION,7
7,2026-03-01,client_73cda7b4e4f265ea,content_2484eda2210ba6e4,0,40.333333,4,REVIEW,LOW_CLICKS_AND_WEAK_POSITION,8
8,2026-03-01,client_73cda7b4e4f265ea,content_41d6b1eaf68d018d,0,76.000000,4,REVIEW,LOW_CLICKS_AND_WEAK_POSITION,9
9,2026-03-01,client_73cda7b4e4f265ea,content_80e5105d7079364d,0,58.666667,4,REVIEW,LOW_CLICKS_AND_WEAK_POSITION,10


In [15]:
import os

os.makedirs("work/outputs", exist_ok=True)

output_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "score",
    "reason_code",
    "action"
]

queue[output_columns].to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV written:", len(queue), "rows")

CSV written: 3611061 rows


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

I reviewed the highest-scoring rows manually. These are decision-support suggestions, not confirmed problems.

For each row, I considered the action, the reason for the score, and what could make the recommendation wrong. For example, low clicks could be normal for a low-volume page, and a weak position does not by itself prove that the content needs to be changed.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = queue.head(20)[[
    "rank",
    "content_hash_id",
    "score",
    "reason_code",
    "action",
    "gsc_clicks",
    "gsc_avg_position"
]]

top20

,rank,content_hash_id,score,reason_code,action,gsc_clicks,gsc_avg_position
0,1,content_476c37c366920c1b,4,LOW_CLICKS_AND_WEAK_POSITION,REVIEW,0,41.000000
1,2,content_f71459b346aba398,4,LOW_CLICKS_AND_WEAK_POSITION,REVIEW,0,38.500000
2,3,content_098eedbd77ec1de1,4,LOW_CLICKS_AND_WEAK_POSITION,REVIEW,0,41.111111
3,4,content_fd377b1af29dcbb3,4,LOW_CLICKS_AND_WEAK_POSITION,REVIEW,0,20.384615
4,5,content_61c495d39498082e,4,LOW_CLICKS_AND_WEAK_POSITION,REVIEW,0,28.894737
5,6,content_097e9ae329f3f6f3,4,LOW_CLICKS_AND_WEAK_POSITION,REVIEW,0,35.400000
6,7,content_76871bb7b214573c,4,LOW_CLICKS_AND_WEAK_POSITION,REVIEW,0,89.000000
7,8,content_2484eda2210ba6e4,4,LOW_CLICKS_AND_WEAK_POSITION,REVIEW,0,40.333333
8,9,content_41d6b1eaf68d018d,4,LOW_CLICKS_AND_WEAK_POSITION,REVIEW,0,76.000000
9,10,content_80e5105d7079364d,4,LOW_CLICKS_AND_WEAK_POSITION,REVIEW,0,58.666667


In [17]:
for _, row in top20.iterrows():
    print(
        f"{int(row['rank'])}. "
        f"Action: {row['action']}. "
        f"Reason: {row['reason_code']}. "
        f"Confidence: directional only. "
        f"Could be wrong if the page is intentionally low-volume, "
        f"new, seasonal, or has a valid reason for its current search position."
    )

1. Action: REVIEW. Reason: LOW_CLICKS_AND_WEAK_POSITION. Confidence: directional only. Could be wrong if the page is intentionally low-volume, new, seasonal, or has a valid reason for its current search position.
2. Action: REVIEW. Reason: LOW_CLICKS_AND_WEAK_POSITION. Confidence: directional only. Could be wrong if the page is intentionally low-volume, new, seasonal, or has a valid reason for its current search position.
3. Action: REVIEW. Reason: LOW_CLICKS_AND_WEAK_POSITION. Confidence: directional only. Could be wrong if the page is intentionally low-volume, new, seasonal, or has a valid reason for its current search position.
4. Action: REVIEW. Reason: LOW_CLICKS_AND_WEAK_POSITION. Confidence: directional only. Could be wrong if the page is intentionally low-volume, new, seasonal, or has a valid reason for its current search position.
5. Action: REVIEW. Reason: LOW_CLICKS_AND_WEAK_POSITION. Confidence: directional only. Could be wrong if the page is intentionally low-volume, new, 

## 4. Weak picks + leakage check

### Weak picks

Some picks may be weak because the rule only uses clicks and average position. A page with low clicks is not necessarily a problem, especially if it is a low-volume or intentionally niche page.

### Leakage check

The baseline uses only March 2026 GSC signals available in the current row. It does not use future-window data, a future label, or a product flag. The score is a simple decision-support baseline and should not be treated as proof that a page needs a content change.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

required_columns = [
    "gsc_clicks",
    "gsc_avg_position"
]

print("Features used:", required_columns)
print("Future-window or label-derived columns used: none")
print("Product flags used: none")
print("Top score:", queue["score"].max())
print("Queue rows:", len(queue))

Features used: ['gsc_clicks', 'gsc_avg_position']
Future-window or label-derived columns used: none
Product flags used: none
Top score: 4
Queue rows: 3611061


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.